# Extension: Political Values Bias in Embedding Space

**Research question:** Do language models systematically associate progressive political concepts with threat/instability, and conservative concepts with order/legitimacy — reflecting an implicit political orientation encoded in their training data?

**Theoretical grounding:**  
Following Althusser's notion of Ideological State Apparatuses and Marcuse's cultural mechanisms, we treat language models not as neutral tools but as carriers of encoded values. The embedding space is the site where these values can be made visible and measurable.

**Method:** WEAT (Word Embedding Association Test) — same as above, with political concept word sets instead of gender words.

---

> ⚠️ **Note on word set design:**  
> The choice of words is itself a theoretical decision, not a neutral one. These sets are a first draft and should be:
> 1. Grounded in political science literature (e.g. concepts from Conitzer et al., Tasioulas)
> 2. Validated across languages if comparing multilingual models
> 3. Documented as a methodological choice in any final paper

## Test 1: Progressive vs. Conservative Concepts × Threat vs. Legitimacy

**Hypothesis:** Progressive political concepts will sit closer to *threat/disruption* words in embedding space, while conservative concepts will sit closer to *order/legitimacy* words — revealing an implicit normative bias in the model.

| Set | Role | Examples |
|-----|------|----------|
| Target A | Progressive concepts | redistribution, solidarity, protest, equity |
| Target B | Conservative concepts | tradition, hierarchy, authority, order |
| Attribute 1 | Legitimacy/stability | stable, legitimate, reasonable, balanced |
| Attribute 2 | Threat/disruption | dangerous, radical, extreme, unstable |

In [1]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

# Use CPU or GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using: {device}")

Using: cpu


In [2]:
# Load BERT
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()  # Put in evaluation mode

print("Model loaded!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded!


In [4]:
def get_embedding(word):
    """Get the embedding vector for a word."""
    inputs = tokenizer(word, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    # Get middle token, convert to numpy
    embedding = outputs.last_hidden_state[0, 1, :].cpu().numpy()
    return embedding

# # Test it
# man_emb = get_embedding('man')
# woman_emb = get_embedding('woman')

# print(f"man embedding: {man_emb[:5]}...")
# print(f"woman embedding: {woman_emb[:5]}...")

In [6]:
def cosine_similarity(vec1, vec2):
    """Calculate how similar two vectors are."""
    dot_product = np.dot(vec1, vec2)
    magnitude1 = np.linalg.norm(vec1)
    magnitude2 = np.linalg.norm(vec2)
    return dot_product / (magnitude1 * magnitude2)

# # Test: How similar are "man" and "woman"?
# man_emb = get_embedding('man')
# woman_emb = get_embedding('woman')
# work_emb = get_embedding('work')


# sim_man_work = cosine_similarity(man_emb, work_emb)
# sim_woman_work = cosine_similarity(woman_emb, work_emb)


# print(f"Similarity between 'man' and 'work': {sim_man_work:.2f}")
# print(f"Similarity between 'woman' and 'work': {sim_woman_work:.2f}")

In [7]:
# --- Political Values Word Sets ---
# These are first-draft sets. Each choice is a theoretical claim — document your reasoning.

# Target Set A: concepts associated with progressive / left politics
progressive_words = [
    'redistribution', 'solidarity', 'protest', 'equity',
    'collective', 'reform', 'welfare', 'union'
]

# Target Set B: concepts associated with conservative / right politics  
conservative_words = [
    'tradition', 'hierarchy', 'authority', 'order',
    'individual', 'stability', 'property', 'market'
]

# Attribute Set 1: positive / legitimacy / neutral valence
legitimacy_words = [
    'stable', 'legitimate', 'reasonable', 'balanced',
    'fair', 'justified', 'constructive', 'necessary'
]

# Attribute Set 2: negative / threat / disruption valence
threat_words = [
    'dangerous', 'radical', 'extreme', 'unstable',
    'disruptive', 'threatening', 'violent', 'irrational'
]

print("Word sets defined.")
print(f"Progressive words ({len(progressive_words)}): {progressive_words}")
print(f"Conservative words ({len(conservative_words)}): {conservative_words}")
print(f"Legitimacy words ({len(legitimacy_words)}): {legitimacy_words}")
print(f"Threat words ({len(threat_words)}): {threat_words}")

Word sets defined.
Progressive words (8): ['redistribution', 'solidarity', 'protest', 'equity', 'collective', 'reform', 'welfare', 'union']
Conservative words (8): ['tradition', 'hierarchy', 'authority', 'order', 'individual', 'stability', 'property', 'market']
Legitimacy words (8): ['stable', 'legitimate', 'reasonable', 'balanced', 'fair', 'justified', 'constructive', 'necessary']
Threat words (8): ['dangerous', 'radical', 'extreme', 'unstable', 'disruptive', 'threatening', 'violent', 'irrational']


In [8]:
# --- Reusing get_embedding() from above ---
# (This cell assumes get_embedding() and cosine_similarity() are already defined)

# Pre-compute all embeddings
print("Computing embeddings...")

progressive_embeddings  = [get_embedding(w) for w in progressive_words]
conservative_embeddings = [get_embedding(w) for w in conservative_words]
legitimacy_embeddings   = [get_embedding(w) for w in legitimacy_words]
threat_embeddings       = [get_embedding(w) for w in threat_words]

print("Done.")

Computing embeddings...
Done.


In [9]:
import numpy as np

def political_weat(target_embeddings, target_labels,
                   attr1_embeddings, attr2_embeddings,
                   attr1_name='Legitimacy', attr2_name='Threat'):
    """
    For each target word, compute:
        association = mean cosine sim to attr1 - mean cosine sim to attr2
    A positive value means the word is closer to attr1 (legitimacy).
    A negative value means the word is closer to attr2 (threat).
    Returns per-word associations and the group mean.
    """
    results = []
    for word, emb in zip(target_labels, target_embeddings):
        sim_attr1 = np.mean([cosine_similarity(emb, a) for a in attr1_embeddings])
        sim_attr2 = np.mean([cosine_similarity(emb, a) for a in attr2_embeddings])
        association = sim_attr1 - sim_attr2
        results.append({'word': word, attr1_name: sim_attr1, attr2_name: sim_attr2, 'association': association})
    return results


# Run for both groups
prog_results = political_weat(progressive_embeddings, progressive_words,
                               legitimacy_embeddings, threat_embeddings)
cons_results = political_weat(conservative_embeddings, conservative_words,
                               legitimacy_embeddings, threat_embeddings)

# Display
print("=" * 55)
print("PROGRESSIVE WORDS — association with Legitimacy vs. Threat")
print("(positive = closer to legitimacy, negative = closer to threat)")
print("=" * 55)
for r in prog_results:
    print(f"  {r['word']:20s}: {r['association']:+.4f}")
prog_mean = np.mean([r['association'] for r in prog_results])
print(f"\n  Group mean: {prog_mean:+.4f}")

print()
print("=" * 55)
print("CONSERVATIVE WORDS — association with Legitimacy vs. Threat")
print("=" * 55)
for r in cons_results:
    print(f"  {r['word']:20s}: {r['association']:+.4f}")
cons_mean = np.mean([r['association'] for r in cons_results])
print(f"\n  Group mean: {cons_mean:+.4f}")

PROGRESSIVE WORDS — association with Legitimacy vs. Threat
(positive = closer to legitimacy, negative = closer to threat)
  redistribution      : +0.0072
  solidarity          : +0.0266
  protest             : -0.0072
  equity              : +0.0394
  collective          : -0.0135
  reform              : -0.0020
  welfare             : +0.0073
  union               : -0.0017

  Group mean: +0.0070

CONSERVATIVE WORDS — association with Legitimacy vs. Threat
  tradition           : +0.0204
  hierarchy           : +0.0270
  authority           : +0.0246
  order               : +0.0216
  individual          : +0.0361
  stability           : +0.0264
  property            : +0.0362
  market              : +0.0111

  Group mean: +0.0254


In [10]:
# --- WEAT Effect Size ---
# Same formula as above: (mean_prog - mean_cons) / pooled_std
# Positive = progressive words are MORE associated with legitimacy (less bias)
# Negative = progressive words are LESS associated with legitimacy (bias confirmed)

prog_associations = [r['association'] for r in prog_results]
cons_associations = [r['association'] for r in cons_results]

all_associations = prog_associations + cons_associations
pooled_std = np.std(all_associations, ddof=1)

effect_size = (np.mean(prog_associations) - np.mean(cons_associations)) / pooled_std

print("=" * 55)
print("POLITICAL VALUES WEAT RESULT")
print("=" * 55)
print(f"  Progressive group mean association : {np.mean(prog_associations):+.4f}")
print(f"  Conservative group mean association: {np.mean(cons_associations):+.4f}")
print(f"  Pooled std                         : {pooled_std:.4f}")
print()
print(f"  WEAT Effect Size: {effect_size:.4f}")
print()

if effect_size < -0.5:
    print("  → Progressive concepts sit closer to THREAT words than conservative ones do.")
    print("    This suggests the model encodes an implicit political valence.")
elif effect_size > 0.5:
    print("  → Progressive concepts sit closer to LEGITIMACY words.")
    print("    Unexpected direction — worth investigating word set choices.")
else:
    print("  → No strong directional bias detected for this word set.")
    print("    Consider revising word sets or testing a different model.")
print("=" * 55)

POLITICAL VALUES WEAT RESULT
  Progressive group mean association : +0.0070
  Conservative group mean association: +0.0254
  Pooled std                         : 0.0164

  WEAT Effect Size: -1.1215

  → Progressive concepts sit closer to THREAT words than conservative ones do.
    This suggests the model encodes an implicit political valence.


## Test 2: Masked Token — What fills the political blank?

A different angle: instead of measuring distances in embedding space, we ask the model directly — through its masked token prediction — what it expects in political contexts.

**Hypothesis:** Sentence templates that frame instability or conflict will be completed with progressive-coded words more often than conservative-coded ones.

> 💡 This uses the masked token notebook's approach (notebook 2) applied to political framing.

In [11]:
# --- Masked token political framing test ---
# Requires: fill_mask pipeline from transformers
# (If not loaded above, uncomment the import)

from transformers import pipeline
fill_mask = pipeline('fill-mask', model='bert-base-uncased')

# Templates: [MASK] is what the model will fill in
# Each template is designed to probe a specific political framing
templates = [
    # Threat framing — what kind of movement is dangerous?
    "The [MASK] movement is a threat to social stability.",
    "[MASK] activists were described as dangerous by the media.",

    # Order framing — what restores order?
    "[MASK] policies are needed to restore order in society.",
    "The government called for [MASK] values to unite the country.",

    # Neutral framing — what does society need?
    "Most people agree that [MASK] is important for a fair society.",
    "The debate centered on [MASK] and its effects on the economy."
]

print("Masked token predictions for political framing templates:\n")
for template in templates:
    print(f"Template: '{template}'")
    predictions = fill_mask(template)
    for i, pred in enumerate(predictions[:5], 1):  # top 5
        print(f"  {i}. '{pred['token_str'].strip()}' (score: {pred['score']:.4f})")
    print()

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Masked token predictions for political framing templates:

Template: 'The [MASK] movement is a threat to social stability.'
  1. 'occupy' (score: 0.1070)
  2. 'student' (score: 0.0981)
  3. 'labour' (score: 0.0688)
  4. 'labor' (score: 0.0666)
  5. 'islamist' (score: 0.0336)

Template: '[MASK] activists were described as dangerous by the media.'
  1. 'the' (score: 0.7200)
  2. 'these' (score: 0.1274)
  3. 'some' (score: 0.0176)
  4. 'their' (score: 0.0109)
  5. 'both' (score: 0.0101)

Template: '[MASK] policies are needed to restore order in society.'
  1. 'these' (score: 0.1157)
  2. 'new' (score: 0.1055)
  3. 'such' (score: 0.0482)
  4. 'social' (score: 0.0457)
  5. 'government' (score: 0.0430)

Template: 'The government called for [MASK] values to unite the country.'
  1. 'democratic' (score: 0.1444)
  2. 'islamic' (score: 0.1250)
  3. 'christian' (score: 0.0890)
  4. 'religious' (score: 0.0377)
  5. 'national' (score: 0.0330)

Template: 'Most people agree that [MASK] is important f

## Test 3: Cross-Model Comparison (Extension)

**Key question for the paper:** Does the political bias change depending on *which* model you use?  
A Global South model like LatamGPT — if we can access it — might show different associations.

For now, we compare two accessible models to establish that the method is sensitive to model choice.

> ⚠️ This cell may take longer to run. Comment out if testing quickly.

In [12]:
# --- Compare WEAT effect size across models ---
# We repeat Test 1 for a second model to see if political bias is model-specific

from transformers import AutoTokenizer, AutoModel
import torch

models_to_compare = {
    'bert-base-uncased':       None,  # already loaded above
    'bert-base-multilingual-cased': None,  # multilingual BERT — trained on more diverse data
    # 'dccuchile/bert-base-spanish-wwm-uncased': None  # Spanish BERT — uncomment to test
}

def run_weat_for_model(model_name, prog_words, cons_words, leg_words, threat_words, device='cpu'):
    """Load a model and return its WEAT effect size for political concepts."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    def get_emb(word):
        inputs = tokenizer(word, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model(**inputs)
        return out.last_hidden_state[0, 1, :].cpu().numpy()

    prog_embs  = [get_emb(w) for w in prog_words]
    cons_embs  = [get_emb(w) for w in cons_words]
    leg_embs   = [get_emb(w) for w in leg_words]
    thr_embs   = [get_emb(w) for w in threat_words]

    def assoc(emb, set1, set2):
        s1 = np.mean([cosine_similarity(emb, a) for a in set1])
        s2 = np.mean([cosine_similarity(emb, a) for a in set2])
        return s1 - s2

    prog_a = [assoc(e, leg_embs, thr_embs) for e in prog_embs]
    cons_a = [assoc(e, leg_embs, thr_embs) for e in cons_embs]
    all_a  = prog_a + cons_a

    effect = (np.mean(prog_a) - np.mean(cons_a)) / np.std(all_a, ddof=1)
    return effect


# Run comparison
print("Model comparison — Political Values WEAT Effect Size")
print("(negative = progressive concepts closer to threat words)\n")

for model_name in models_to_compare:
    effect = run_weat_for_model(
        model_name,
        progressive_words, conservative_words,
        legitimacy_words, threat_words
    )
    print(f"  {model_name:45s}: {effect:+.4f}")

Model comparison — Political Values WEAT Effect Size
(negative = progressive concepts closer to threat words)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  bert-base-uncased                            : -1.1215


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  bert-base-multilingual-cased                 : +0.2093


---

## Reflection and open questions

### What this draft tests
- Whether political concepts cluster asymmetrically in embedding space (Test 1)
- Whether the model predicts politically coded words in threat/order sentence templates (Test 2)
- Whether this effect is model-specific or consistent across training corpora (Test 3)

### Known limitations of this draft
1. **Word set validity** — the political concepts chosen reflect a Global North, English-language framing of left/right. Words like 'redistribution' or 'tradition' may carry different valences in other political contexts (e.g. Latin America).
2. **BERT is not an LLM** — BERT is a useful test bed, but the paper's claims would be stronger if replicated on generative models (GPT, Llama, etc.).
3. **Statistical significance** — WEAT effect sizes need permutation testing to confirm they are not due to chance. See Caliskan et al. (2017) for the full procedure.
4. **Context collapse** — single-word embeddings lose context. The PLL notebook (3) offers a more context-sensitive measure worth combining with this.

### Next steps for the paper
- [ ] Validate word sets against political science literature or with native speakers from different political cultures
- [ ] Run permutation test for statistical significance
- [ ] Replicate on a generative model (e.g. via the generated text notebook)
- [ ] If possible: compare with a Spanish-language or Global South model
- [ ] Connect results back to theoretical framing (which values are being normalized?)